# Plant Disease Detection: Complete End-to-End Project

This notebook demonstrates the entire workflow:
1. **Data Loading & Exploration** - Load PlantVillage dataset
2. **Model Training** - Train CNN v1 & v2 with MLflow tracking
3. **Model Evaluation** - Evaluate on test set
4. **Inference** - Make predictions on new images
5. **Deployment Demo** - Show how FastAPI serves the model

## Part 1: Setup & Imports

In [ ]:
import os
import sys
from pathlib import Path
import json
import random
import time

# Add project root to path
ROOT_DIR = Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

# Scientific computing
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import seaborn as sns

# Deep learning
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms, datasets
from torch.utils.data import DataLoader, Subset

# MLflow tracking
import mlflow

# Project imports
from src.data_loader import get_dataloaders, get_transforms
from src.model import get_model
from src.evaluate import evaluate_model
from deployment.utils import preprocess_image, predict_image, load_model as load_model_util

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")
print(f"Project root: {ROOT_DIR}")

## Part 2: Configure MLflow & Dataset Paths

In [ ]:
# Configuration
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MODELS_DIR = ROOT_DIR / 'models'
MLRUNS_DIR = ROOT_DIR / 'mlruns'
DATA_DIR = Path(r"C:\Users\hasna\Downloads\PlantVillage\PlantVillage")  # Update as needed

# Create directories
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Configure MLflow
os.environ['MLFLOW_ALLOW_FILE_STORE'] = 'true'
mlflow.set_tracking_uri(MLRUNS_DIR.resolve().as_uri())

print(f"Data directory: {DATA_DIR}")
print(f"Models directory: {MODELS_DIR}")
print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")
print(f"Dataset exists: {DATA_DIR.exists()}")

## Part 3: Load & Explore Dataset

In [ ]:
# Load dataloaders
batch_size = 32
train_loader, val_loader, test_loader, class_names = get_dataloaders(
    data_dir=str(DATA_DIR),
    batch_size=batch_size,
)

print(f"Number of classes: {len(class_names)}")
print(f"Classes: {class_names}")
print(f"\nDataLoader info:")
print(f"  Train batches: {len(train_loader)} (samples: {len(train_loader.dataset)})")
print(f"  Val batches: {len(val_loader)} (samples: {len(val_loader.dataset)})")
print(f"  Test batches: {len(test_loader)} (samples: {len(test_loader.dataset)})")

## Part 4: Visualize Sample Images

In [ ]:
# Get a sample batch
images, labels = next(iter(train_loader))

# Denormalize for visualization
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i, (img, label) in enumerate(zip(images[:8], labels[:8])):
    ax = axes[i // 4, i % 4]
    img_denorm = (img * std + mean).clamp(0, 1).permute(1, 2, 0)
    ax.imshow(img_denorm.numpy())
    ax.set_title(class_names[label])
    ax.axis('off')

plt.suptitle('Sample Training Images')
plt.tight_layout()
plt.show()

print(f"Sample batch shape: {images.shape}")
print(f"Sample labels: {[class_names[l] for l in labels[:8]]}")

## Part 5: Train Model v1 (Baseline CNN)

In [ ]:
# Set seed for reproducibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

# Create model v1
model_v1 = get_model(version='v1', num_classes=len(class_names)).to(DEVICE)
print(f"Model v1 created")
print(f"Model parameters: {sum(p.numel() for p in model_v1.parameters()) / 1e6:.2f}M")

# Training setup
criterion = nn.CrossEntropyLoss()
optimizer_v1 = optim.Adam(model_v1.parameters(), lr=0.001)
num_epochs = 5  # Reduced for demo; use 15 for full training

print(f"\nStarting v1 training on {DEVICE}...")
print(f"Epochs: {num_epochs}, Batch size: {batch_size}")

In [ ]:
# Training loop for v1
train_losses_v1 = []
val_losses_v1 = []
train_accs_v1 = []
val_accs_v1 = []
best_val_acc_v1 = 0.0
best_model_path_v1 = MODELS_DIR / 'plant_disease_v1.pth'

mlflow.set_experiment('plant_disease_detection')

with mlflow.start_run(run_name=f'v1_{int(time.time())}') as run:
    mlflow.log_param('version', 'v1')
    mlflow.log_param('learning_rate', 0.001)
    mlflow.log_param('optimizer', 'Adam')
    mlflow.log_param('epochs', num_epochs)
    mlflow.log_param('batch_size', batch_size)
    
    for epoch in range(1, num_epochs + 1):
        # Training phase
        model_v1.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0
        
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            optimizer_v1.zero_grad()
            outputs = model_v1(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer_v1.step()
            
            train_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            train_correct += (preds == labels).sum().item()
            train_total += labels.size(0)
        
        train_loss_avg = train_loss / train_total
        train_acc = 100.0 * train_correct / train_total
        train_losses_v1.append(train_loss_avg)
        train_accs_v1.append(train_acc)
        
        # Validation phase
        val_loss, val_acc = evaluate_model(model_v1, val_loader, criterion, DEVICE)
        val_losses_v1.append(val_loss)
        val_accs_v1.append(val_acc)
        
        # MLflow logging
        mlflow.log_metric('train_loss', train_loss_avg, step=epoch)
        mlflow.log_metric('val_loss', val_loss, step=epoch)
        mlflow.log_metric('train_accuracy', train_acc, step=epoch)
        mlflow.log_metric('val_accuracy', val_acc, step=epoch)
        
        print(f'Epoch [{epoch}/{num_epochs}] Train Loss: {train_loss_avg:.4f} | Train Acc: {train_acc:.2f}% | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%')
        
        # Save best model
        if val_acc > best_val_acc_v1:
            best_val_acc_v1 = val_acc
            torch.save(model_v1.state_dict(), best_model_path_v1)
            print(f'  -> Best model saved: {best_model_path_v1}')
    
    # Test phase
    test_loss, test_acc = evaluate_model(model_v1, test_loader, criterion, DEVICE)
    mlflow.log_metric('test_loss', test_loss)
    mlflow.log_metric('test_accuracy', test_acc)
    
    print(f'\nV1 Training Complete:')
    print(f'  Best Val Accuracy: {best_val_acc_v1:.2f}%')
    print(f'  Test Accuracy: {test_acc:.2f}%')

## Part 6: Visualize Training Metrics

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss plot
axes[0].plot(train_losses_v1, label='Train Loss', marker='o')
axes[0].plot(val_losses_v1, label='Val Loss', marker='s')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('V1 Training Loss')
axes[0].legend()
axes[0].grid(True)

# Accuracy plot
axes[1].plot(train_accs_v1, label='Train Accuracy', marker='o')
axes[1].plot(val_accs_v1, label='Val Accuracy', marker='s')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('V1 Training Accuracy')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

## Part 7: Load Saved Model & Make Predictions

In [ ]:
# Load class names
class_names_path = MODELS_DIR / 'class_names.json'
if class_names_path.exists():
    with open(class_names_path, 'r') as f:
        saved_class_names = json.load(f)
    print(f"Loaded class names: {len(saved_class_names)} classes")
else:
    # Save class names for future use
    with open(class_names_path, 'w') as f:
        json.dump(class_names, f, indent=2)
    saved_class_names = class_names
    print(f"Saved class names: {len(saved_class_names)} classes")

# Load trained model
loaded_model = get_model(version='v1', num_classes=len(saved_class_names)).to(DEVICE)
loaded_model.load_state_dict(torch.load(best_model_path_v1, map_location=DEVICE))
loaded_model.eval()
print(f"Model loaded from {best_model_path_v1}")

## Part 8: Inference on Test Images

In [ ]:
# Get batch from test loader
test_images, test_labels = next(iter(test_loader))
test_images = test_images.to(DEVICE)

# Make predictions
with torch.no_grad():
    outputs = loaded_model(test_images)
    probabilities = torch.softmax(outputs, dim=1)
    preds = torch.argmax(probabilities, dim=1)
    confidences = torch.max(probabilities, dim=1).values

# Visualize predictions
fig, axes = plt.subplots(2, 4, figsize=(14, 7))

for i in range(8):
    ax = axes[i // 4, i % 4]
    
    # Denormalize image
    img = test_images[i].cpu()
    img_denorm = (img * std + mean).clamp(0, 1).permute(1, 2, 0)
    ax.imshow(img_denorm.numpy())
    
    # Prediction
    pred_label = saved_class_names[preds[i]]
    true_label = saved_class_names[test_labels[i]]
    confidence = confidences[i].item() * 100
    
    # Color based on correctness
    color = 'green' if preds[i] == test_labels[i] else 'red'
    ax.set_title(f'Pred: {pred_label}\nConf: {confidence:.1f}%', color=color, fontweight='bold')
    ax.axis('off')

plt.suptitle('Predictions on Test Images')
plt.tight_layout()
plt.show()

# Calculate accuracy on batch
batch_acc = (preds == test_labels.to(DEVICE)).sum().item() / len(test_labels) * 100
print(f"Batch Accuracy: {batch_acc:.2f}%")

## Part 9: FastAPI Deployment Demo

In [ ]:
print("FastAPI Deployment Setup:")
print("="*60)
print(f"\n1. Model saved at: {best_model_path_v1}")
print(f"2. Class names saved at: {class_names_path}")
print(f"3. Deployment code at: deployment/app.py")
print(f"\n4. Run the FastAPI server:")
print(f"   Windows: .venv\\Scripts\\python.exe deployment\\app.py")
print(f"   Linux/Mac: python deployment/app.py")
print(f"\n5. Endpoints:")
print(f"   - GET  http://localhost:8000/")
print(f"   - GET  http://localhost:8000/health")
print(f"   - POST http://localhost:8000/predict (upload image)")
print(f"   - Docs: http://localhost:8000/docs")
print("="*60)

## Part 10: Summary & Next Steps

In [ ]:
summary = f"""
PROJECT COMPLETE!
================

Training Results:
  - Best Validation Accuracy (v1): {best_val_acc_v1:.2f}%
  - Test Accuracy (v1): {test_acc:.2f}%
  - Model saved: {best_model_path_v1}
  - Class names saved: {class_names_path}

Next Steps:
  1. To run v2 (improved model), execute: python src/train.py --version v2 --data-dir \"{DATA_DIR}\"
  2. To deploy as API, run: python deployment/app.py
  3. To build Docker image, run: docker build -t dl-lab-app .
  4. To run Docker container, run: docker run -p 8000:8000 dl-lab-app

Directories:
  - Models: {MODELS_DIR}
  - MLflow tracking: {MLRUNS_DIR}
  - Deployment code: {ROOT_DIR / 'deployment'}
  - Training code: {ROOT_DIR / 'src'}

For production GPU training, use Ubuntu/WSL with CUDA support.
"""

print(summary)